In [1]:
import ftplib
import io 
import os

import pandas as pd

import plotly.graph_objects as go

from dotenv import load_dotenv

In [2]:
# Load .env file 
load_dotenv()

True

In [3]:
host = os.getenv('AZURE_FTP_HOST')
user = os.getenv('AZURE_FTP_USER')
passwd = os.getenv('AZURE_FTP_PWD')

In [34]:
ftp = ftplib.FTP(host=host, user=user, passwd=passwd)

dataframes = []

# Get all files stored in folder
files = ftp.nlst('/TDS1')

for filename in files:
    if filename.endswith(".tru"):
        csv_buffer = io.BytesIO()
        ftp.retrbinary(f"RETR {filename}", csv_buffer.write)
        csv_buffer.seek(0)

        dataframes.append(pd.read_csv(csv_buffer,
                                      header=0, 
                                      names=['Timestamp', 
                                             'Temp_15', 
                                             'Temp_50', 
                                             'Temp_75',
                                             'VMC_15', 
                                             'VMC_50', 
                                             'VMC_75'
                                      ],
                                      skiprows=1, 
                                      skipfooter=1, 
                                      engine='python'
                                      )
        )

df_TDS1 = pd.concat(dataframes)


In [115]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_TDS1['Timestamp'].astype('datetime64[ns]'),
        y=df_TDS1['VMC_15'], 
        mode='markers', 
        name='15cm'
    )
)
fig.add_trace(
    go.Scatter(
        x=df_TDS1['Timestamp'].astype('datetime64[ns]'),
        y=df_TDS1['VMC_50'], 
        mode='markers', 
        name='50cm'
    )
)
fig.add_trace(
    go.Scatter(
        x=df_TDS1['Timestamp'].astype('datetime64[ns]'),
        y=df_TDS1['VMC_75'], 
        mode='markers', 
        name='75cm'
    )
)
fig.update_layout(
    template='simple_white',
    #plot_bgcolor='rgba(0,0,0,0)',
    #paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(
        title='Timpstamp',
    ),
    yaxis=dict(
        title='Volumetric water content [%]',
        side='left', 
        range=[0, 100]
    ),
    legend=dict(
        xanchor='left', 
        x = 0.01, 
        yanchor='top', 
        y = 0.3
    )
)

In [116]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_TDS1['Timestamp'].astype('datetime64[ns]'),
        y=df_TDS1['Temp_15'], 
        mode='markers', 
        name='15cm'
    )
)
fig.add_trace(
    go.Scatter(
        x=df_TDS1['Timestamp'].astype('datetime64[ns]'),
        y=df_TDS1['Temp_50'], 
        mode='markers', 
        name='50cm'
    )
)
fig.add_trace(
    go.Scatter(
        x=df_TDS1['Timestamp'].astype('datetime64[ns]'),
        y=df_TDS1['Temp_75'], 
        mode='markers', 
        name='75cm'
    )
)

fig.update_layout(
    template='simple_white',
    #plot_bgcolor='rgba(0,0,0,0)',
    #paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(
        title='Timpstamp',
    ),
    yaxis=dict(
        title='Temperature [deg C]',
        side='left', 
    ),
    legend=dict(
        xanchor='left', 
        x = 0.01, 
        yanchor='top', 
        y = 0.99
    )
)